# 🌍 WeatherBench 딥러닝 기상 예측 — 입문편
## Z500 24시간 예측 with MLP

| 항목 | 내용 |
|------|------|
| **데이터** | WeatherBench2 ERA5 (GCS zarr 스트리밍) |
| **변수** | Z500 (500hPa 지위고도) |
| **입력** | t 시각의 전구 Z500 필드 |
| **출력** | t+24h의 전구 Z500 필드 |
| **모델** | MLP |
| **Train** | 2015–2016 · Val: 2017 · Test: 2018 |

---

## 목차
1. [WeatherBench 소개](#소개)
2. [환경 설정 및 라이브러리 임포트](#환경)
3. [설정값 정의](#설정)
4. [데이터 로드 (GCS Zarr)](#데이터)
5. [탐색적 데이터 분석 (EDA)](#EDA)
6. [전처리 및 데이터셋 구성](#전처리)
7. [MLP 모델 정의](#모델)
8. [학습](#학습)
9. [평가 및 시각화](#평가)
10. [요약 및 다음 단계](#요약)

## 1. WeatherBench 소개 <a name="소개"></a>

**WeatherBench**는 딥러닝 기반 날씨 예측 모델의 표준 벤치마크 데이터셋입니다.  
Rasp et al. (2020) 논문에서 제안되었으며, ERA5 재분석 데이터를 기반으로 합니다.

### WeatherBench2란?
- **WeatherBench2**는 원본의 업그레이드 버전으로, 더 높은 해상도와 더 긴 시계열을 제공합니다.
- Google Cloud Storage(GCS)에 공개 Zarr 형식으로 저장되어 있어 스트리밍 로드가 가능합니다.
- 전 세계 연구자들이 동일한 데이터와 평가 기준을 사용할 수 있도록 설계되었습니다.

### 이 노트북에서 다루는 내용
- **변수**: Z500 — 500 hPa 등압면에서의 지위고도(Geopotential Height)
  - 대기 중층의 흐름을 나타내는 핵심 변수로, 중위도 날씨 패턴과 밀접한 관련이 있습니다.
- **모델**: MLP(Multi-Layer Perceptron) — 딥러닝의 가장 기초적인 모델
- **목표**: 현재 Z500 전구 필드 → 24시간 후 Z500 전구 필드 예측

### 평가 지표
- **RMSE** (Root Mean Square Error): 예측 오차의 크기를 나타냅니다.
- **Persistence Baseline**: "미래는 현재와 같다"는 가장 단순한 예측 전략. 이를 이겨야 의미 있는 모델입니다.

> 📌 **참고문헌**  
> Rasp, S., et al. (2020). WeatherBench: A benchmark dataset for data-driven weather forecasting. *JAMES*, 12(11).  
> Rasp, S., et al. (2024). WeatherBench 2: A benchmark for the next generation of data-driven global weather models. *JAMES*, 16(6).

In [ ]:
# Colab 환경 설정
!pip install -q gcsfs zarr "xarray[io]"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/Colab Notebooks/weather-climate-ai-tutorials

In [ ]:
# 나눔 고딕 폰트 설치
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import gcsfs
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pathlib, matplotlib, matplotlib.font_manager as fm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__} | Device: {DEVICE}')

# ── 한글 폰트 설치 및 등록 ───────────────────────────────────────────
font_path = pathlib.Path(matplotlib.get_data_path()) / 'fonts' / 'ttf' / 'NanumGothic.ttf'

if not font_path.exists():
    print('NanumGothic 폰트 다운로드 중...')
    try:
        urllib.request.urlretrieve(
            'https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf',
            font_path
        )
        print('다운로드 완료')
    except Exception as e:
        print(f'다운로드 실패: {e}')

if font_path.exists():
    fm.fontManager.addfont(str(font_path))
    _font_name = fm.FontProperties(fname=str(font_path)).get_name()
    plt.rcParams['font.family'] = _font_name
    print(f'폰트 등록 완료: {_font_name}')
else:
    for sys_path in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf']:
        if pathlib.Path(sys_path).exists():
            fm.fontManager.addfont(sys_path)
            plt.rcParams['font.family'] = fm.FontProperties(fname=sys_path).get_name()
            print(f'시스템 폰트 사용: {sys_path}')
            break
    else:
        print('한글 폰트 없음 → 기본 폰트 사용')

In [ ]:
# ============================================================
# 설정값 (Configuration)
# ============================================================

# GCS 데이터 경로 (WeatherBench2 ERA5, 1.5도 해상도, 6시간 간격)
GCS_PATH = (
    'gs://weatherbench2/datasets/era5/'
    '1959-2023_01_10-6h-240x121_equiangular_with_poles_conservative.zarr'
)

# 리드타임: 4 스텝 × 6시간 = 24시간 예측
LEAD_STEPS = 4

# 공간 해상도 축소: 4배 coarsen → 240/4=60 lon, 121/4≈30 lat
COARSEN = 4

# 학습 하이퍼파라미터
BATCH_SIZE = 64
EPOCHS     = 50
LR         = 1e-3

# 훈련/검증/테스트 연도 (빠른 실험을 위해 기간을 단축합니다)
TRAIN_YEARS = slice('2015', '2015') # 2015년 한 해만 학습에 사용
VAL_YEAR    = '2016-01'           # 2016년 1월 데이터만 검증에 사용
TEST_YEAR   = '2017-01'           # 2017년 1월 데이터만 테스트에 사용

print('설정값 로드 완료')
print(f'  리드타임: {LEAD_STEPS} 스텝 = {LEAD_STEPS * 6}시간')
print(f'  공간 축소: {COARSEN}배 (약 {240 // COARSEN} × {121 // COARSEN} 격자)')
print(f'  학습 기간: {TRAIN_YEARS.start}–{TRAIN_YEARS.stop} / 검증: {VAL_YEAR} / 테스트: {TEST_YEAR}')

In [ ]:
import dask

# ============================================================
# GCS Zarr에서 데이터 로드
# ============================================================

# 익명 접근으로 GCS 파일시스템 연결
fs = gcsfs.GCSFileSystem(token='anon')
ds = xr.open_zarr(fs.get_mapper(GCS_PATH), consolidated=True)
print(ds)

# Z500 선택 및 시간 슬라이싱 (lazy — 아직 실제 다운로드 안 함)
z500_raw = ds['geopotential'].sel(level=500)

# TRAIN_YEARS의 시작부터 TEST_YEAR까지의 데이터를 로드하도록 수정
# slice 객체는 시작과 끝을 포함합니다.
z500_all = z500_raw.sel(time=slice(TRAIN_YEARS.start, TEST_YEAR))

print(f'\nZ500 shape (lazy): {z500_all.shape}')

# 4배 공간 coarsen (240×121 → ~60×30)
# boundary='trim': 나머지 격자 잘라내기
z500_c = z500_all.coarsen(latitude=COARSEN, longitude=COARSEN, boundary='trim').mean()
print(f'Coarsened shape: {z500_c.shape}')

# Dask 진행 바 활성화
with dask.config.set(scheduler='threads', get=dask.get, progressbar=True):
    # GCS에서 실제 데이터 다운로드 (처음 한 번만 시간이 걸림)
    print('\n데이터 로드 중... (진행 바를 확인하세요)')
    z500_c = z500_c.load()
print('완료!')

In [ ]:
print(z500_c)

In [ ]:
# ============================================================
# 탐색적 데이터 분석 (EDA): 계절별 평균 Z500 지도
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 7))
seasons = {'DJF': [12, 1, 2], 'MAM': [3, 4, 5], 'JJA': [6, 7, 8], 'SON': [9, 10, 11]}
lats = z500_c.latitude.values
lons = z500_c.longitude.values

for ax, (name, months) in zip(axes.flat, seasons.items()):
    mask = z500_c.time.dt.month.isin(months)
    # m²/s² → gpm (geopotential metre) 변환: 나누기 g=9.80665
    mean_field = z500_c.sel(time=mask).mean('time').values / 9.80665
    cf = ax.pcolormesh(lons, lats, mean_field.T, cmap='RdYlBu_r', shading='nearest') # 'shading' 인자 수정 및 mean_field.T 추가
    plt.colorbar(cf, ax=ax, label='Z500 (gpm)')
    ax.set_title(f'{name} 평균 Z500')
    ax.set_xlabel('경도')
    ax.set_ylabel('위도')

plt.suptitle('계절별 평균 Z500 (2015–2018)', fontsize=14)
plt.tight_layout()
plt.savefig('wb_z500_seasonal.png', dpi=120)
plt.show()
print('그림 저장: wb_z500_seasonal.png')

## 2. 전처리 <a name="전처리"></a>

### 훈련/검증/테스트 분할
시계열 데이터이므로 **무조건 시간순으로 분할**해야 합니다.  
미래 데이터가 학습에 포함되는 **데이터 누수(data leakage)**를 방지하기 위함입니다.

| 분할 | 연도 | 목적 |
|------|------|------|
| Train | 2015–2016 | 모델 파라미터 최적화 |
| Val   | 2017      | 하이퍼파라미터 튜닝, 조기 종료 |
| Test  | 2018      | 최종 성능 평가 (학습 완료 후 1회만 사용) |

### 정규화 (Normalization)
- **Z-score 정규화**: $x' = (x - \mu) / \sigma$
- 평균($\mu$)과 표준편차($\sigma$)는 **훈련셋**에서만 계산합니다.
- 검증/테스트셋에는 훈련셋의 통계를 그대로 적용합니다 (미래 정보 차단).

### 입출력 쌍 구성
- 입력: 시각 $t$의 Z500 필드
- 출력: 시각 $t + \text{LEAD\_STEPS}$의 Z500 필드 (= t+24h)

In [ ]:
# ============================================================
# 훈련/검증/테스트 분할 + Z-score 정규화
# ============================================================

# 연도 기반 시간 마스크 생성
mask_tr = z500_c.time.dt.year == int(TRAIN_YEARS.start) # TRAIN_YEARS는 2015년 한 해만
mask_vl = z500_c.time.dt.strftime('%Y-%m') == VAL_YEAR # 2016년 1월만
mask_te = z500_c.time.dt.strftime('%Y-%m') == TEST_YEAR # 2017년 1월만

# numpy 배열로 변환 (N, lat, lon)
z_tr = z500_c.sel(time=mask_tr).values
z_vl = z500_c.sel(time=mask_vl).values
z_te = z500_c.sel(time=mask_te).values

# 훈련셋 통계로 정규화 (미래 정보 차단)
mean_z = z_tr.mean()
std_z  = z_tr.std()
z_tr = (z_tr - mean_z) / std_z
z_vl = (z_vl - mean_z) / std_z
z_te = (z_te - mean_z) / std_z

print(f'Train: {z_tr.shape}, Val: {z_vl.shape}, Test: {z_te.shape}')
print(f'정규화 통계 — 평균: {mean_z:.2f} m2/s2, 표준편차: {std_z:.2f} m2/s2')

In [ ]:
# ============================================================
# (X, y) 입출력 쌍 생성 (리드타임 적용)
# ============================================================

LEAD = LEAD_STEPS  # 4 스텝 = 24시간

def make_pairs(data, lead):
    """시각 t (X)와 t+lead (y)의 쌍을 생성합니다."""
    X = data[:-lead]   # 시각 t
    y = data[lead:]    # 시각 t + lead
    return X.astype(np.float32), y.astype(np.float32)

X_tr, y_tr = make_pairs(z_tr, LEAD)
X_vl, y_vl = make_pairs(z_vl, LEAD)
X_te, y_te = make_pairs(z_te, LEAD)

print(f'X_tr: {X_tr.shape}, y_tr: {y_tr.shape}')
print(f'X_vl: {X_vl.shape}, y_vl: {y_vl.shape}')
print(f'X_te: {X_te.shape}, y_te: {y_te.shape}')

In [ ]:
# ============================================================
# PyTorch Dataset & DataLoader 구성
# ============================================================

class WeatherDataset(Dataset):
    """Z500 (X, y) 쌍을 PyTorch 텐서로 제공하는 데이터셋."""
    def __init__(self, X, y):
        # (N, lat, lon) → (N, 1, lat, lon): 채널 차원 추가
        self.X = torch.tensor(X).unsqueeze(1)
        self.y = torch.tensor(y).unsqueeze(1)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.y[i]

tr_loader = DataLoader(WeatherDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
vl_loader = DataLoader(WeatherDataset(X_vl, y_vl), batch_size=BATCH_SIZE)
te_loader = DataLoader(WeatherDataset(X_te, y_te), batch_size=BATCH_SIZE)

_, lat, lon = X_tr.shape
N_GRID = lat * lon
print(f'격자 크기: lat={lat}, lon={lon}, 총 {N_GRID}개 격자점')
print(f'Train 배치 수: {len(tr_loader)}, Val 배치 수: {len(vl_loader)}')


## 3. MLP 모델 <a name="모델"></a>

### MLP (Multi-Layer Perceptron)

MLP는 완전연결층(Fully Connected Layer)으로만 구성된 가장 기본적인 신경망입니다.  
기상 예측에 MLP를 적용할 때는 전구 필드를 1차원 벡터로 **펼쳐서(flatten)** 입력합니다.

```
입력 (1 × lat × lon) → Flatten → Linear → BN → GELU → Dropout
                               → Linear → BN → GELU → Dropout
                               → Linear → BN → GELU
                               → Linear → Reshape → 출력 (1 × lat × lon)
```

### 사용된 기법
| 기법 | 목적 |
|------|------|
| **BatchNorm** | 학습 안정화, 수렴 속도 향상 |
| **GELU** | ReLU보다 매끄러운 활성화 함수 |
| **Dropout(0.1)** | 과적합 방지 |
| **ReduceLROnPlateau** | 검증 손실이 개선되지 않으면 학습률 감소 |

### MLP의 한계
- 공간적 구조(패턴)를 직접 학습하기 어렵습니다 → CNN/ResNet이 유리
- 격자 수가 많아지면 파라미터 폭발 문제 발생
- 주기성(경도 0°와 360°의 연속성)을 처리하지 못합니다

In [ ]:
# ============================================================
# MLP 모델 정의
# ============================================================

class MLP(nn.Module):
    """전구 Z500 필드 예측을 위한 Multi-Layer Perceptron."""
    def __init__(self, n_grid, hidden=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_grid, hidden),
            nn.BatchNorm1d(hidden), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(hidden, hidden),
            nn.BatchNorm1d(hidden), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(hidden, hidden // 2),
            nn.BatchNorm1d(hidden // 2), nn.GELU(),
            nn.Linear(hidden // 2, n_grid),
        )
        self._lat = None
        self._lon = None

    def forward(self, x):
        B, C, H, W = x.shape
        self._lat, self._lon = H, W
        out = self.net(x.view(B, -1))
        return out.view(B, C, H, W)

model = MLP(N_GRID).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'MLP 파라미터 수: {n_params:,}')
print(model)

In [ ]:
# ============================================================
# 학습 루프 정의 및 실행
# ============================================================

def train(model, tr_loader, vl_loader, epochs=50, lr=1e-3):
    """모델을 훈련하고 훈련/검증 손실 이력을 반환합니다."""
    opt   = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=7, factor=0.5)
    crit  = nn.MSELoss()
    tr_hist, vl_hist = [], []

    for ep in range(1, epochs + 1):
        # --- 훈련 단계 ---
        model.train()
        tr_l = 0.0
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            opt.step()
            tr_l += loss.item() * len(xb)
        tr_l /= len(tr_loader.dataset)

        # --- 검증 단계 ---
        model.eval()
        vl_l = 0.0
        with torch.no_grad():
            for xb, yb in vl_loader:
                vl_l += crit(model(xb.to(DEVICE)), yb.to(DEVICE)).item() * len(xb)
        vl_l /= len(vl_loader.dataset)

        sched.step(vl_l)
        tr_hist.append(tr_l)
        vl_hist.append(vl_l)

        if ep % 10 == 0:
            current_lr = opt.param_groups[0]['lr']
            print(f'[{ep:3d}/{epochs}] train={tr_l:.4f}  val={vl_l:.4f}  lr={current_lr:.1e}')

    return tr_hist, vl_hist

print('MLP 학습 시작...')
tr_hist, vl_hist = train(model, tr_loader, vl_loader, epochs=EPOCHS, lr=LR)

# 학습 곡선 시각화
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(tr_hist, label='Train')
ax.plot(vl_hist, label='Val')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss (정규화)')
ax.set_title('학습 곡선')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('wb_loss.png', dpi=120)
plt.show()

In [ ]:
# ============================================================
# 평가: RMSE vs. Persistence Baseline
# ============================================================

def predict_all(model, loader):
    """DataLoader 전체에 대한 예측을 수집합니다."""
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for xb, yb in loader:
            preds.append(model(xb.to(DEVICE)).cpu().numpy())
            trues.append(yb.numpy())
    return np.concatenate(preds), np.concatenate(trues)

pred, true = predict_all(model, te_loader)

# 역정규화 (m²/s² 단위로 복원)
pred_orig = pred * std_z + mean_z
true_orig = true * std_z + mean_z

# MLP RMSE
rmse_mlp = np.sqrt(np.mean((pred_orig - true_orig) ** 2))

# Persistence 기준선: 입력(X_te)을 그대로 미래 예측으로 사용
persist_orig = X_te * std_z + mean_z
rmse_pers = np.sqrt(np.mean((persist_orig - true_orig) ** 2))

print(f"{'모델':<15} {'RMSE (m2/s2)':>15}")
print('-' * 32)
print(f"{'Persistence':<15} {rmse_pers:>15.2f}")
print(f"{'MLP':<15} {rmse_mlp:>15.2f}")
print(f'\nMLP 개선율: {(1 - rmse_mlp / rmse_pers) * 100:.1f}%')

In [ ]:
# ============================================================
# 예측 시각화: 실제 / 예측 / 오차 지도
# ============================================================

idx = 100  # 시각화할 테스트 샘플 인덱스
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
lats = z500_c.latitude.values
lons = z500_c.longitude.values
vmin, vmax = true_orig[idx, 0].min(), true_orig[idx, 0].max()

plot_data = [
    (true_orig[idx, 0],                          '실제 Z500 (t+24h)'),
    (pred_orig[idx, 0],                          'MLP 예측 Z500'),
    (pred_orig[idx, 0] - true_orig[idx, 0],      '오차 (예측-실제)'),
]

for ax, (data, title) in zip(axes, plot_data):
    is_error = '오차' in title
    cmap = 'RdBu_r' if is_error else 'RdYlBu_r'
    vn   = -200 if is_error else vmin
    vx   =  200 if is_error else vmax
    cf = ax.pcolormesh(lons, lats, data.T, cmap=cmap, vmin=vn, vmax=vx, shading='nearest') # data.T 추가 및 shading='nearest'로 수정
    plt.colorbar(cf, ax=ax, label='m2/s2')
    ax.set_title(title)
    ax.set_xlabel('경도')
    ax.set_ylabel('위도')

plt.suptitle('Z500 +24h 예측 예시 (2018년)', fontsize=13)
plt.tight_layout()
plt.savefig('wb_prediction_example.png', dpi=120)
plt.show()

## 4. 요약 및 다음 단계 <a name="요약"></a>

### 이 노트북에서 배운 것

1. **WeatherBench2 데이터** — GCS Zarr 스트리밍으로 ERA5 데이터를 효율적으로 로드하는 방법
2. **기상 데이터 전처리** — 시간순 분할, Z-score 정규화, 리드타임 쌍 구성
3. **MLP 모델** — PyTorch를 이용한 기본적인 딥러닝 날씨 예측 파이프라인
4. **Persistence Baseline** — 딥러닝 모델 성능의 최소 기준

### 한계점

| 한계 | 해결 방향 |
|------|----------|
| 공간 구조 무시 (flatten) | CNN, ResNet 사용 |
| 단일 변수 (Z500만) | 다변수 입력 (T850, U10, V10 등) |
| 단일 리드타임 (+24h만) | 다중 리드타임 예측 |
| 위도 가중치 미적용 | 위도 코사인 가중 RMSE 사용 |
| 경도 주기성 미처리 | Circular padding |

### 다음 단계 (중급편 노트북)

- **ResNet** 기반 모델로 Z500 + T850 **다변수** 예측
- **다중 리드타임** (+24h / +72h / +120h) 동시 학습
- **ACC (Anomaly Correlation Coefficient)** 평가 지표 추가
- **Cartopy** 기반 전구 오차 지도 시각화

### 더 읽을거리

- [WeatherBench2 GitHub](https://github.com/google-research/weatherbench2)
- [WeatherBench2 논문 (Rasp et al., 2024)](https://agupubs.onlinelibrary.wiley.com/doi/10.1029/2023MS003854)
- [Pangu-Weather (Bi et al., 2023)](https://www.nature.com/articles/s41586-023-06185-3)
- [GraphCast (Lam et al., 2023)](https://www.science.org/doi/10.1126/science.adi2336)